# EdgeTRM (Official TRM path, notebook-friendly)

This notebook uses the **official TinyRecursiveModels TRM + ACT loss** through a thin adapter, instead of re-implementing recursion in-notebook.

Goal: provide a small, runnable baseline where loss decreases reliably, then you can swap in Sudoku/ARC data.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

In [ ]:
import torch
from edge_trm_wrapper import EdgeTRMAdapter, EdgeTRMBatch

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1) Tiny config (fast sanity training)

This is intentionally tiny and uses `halt_max_steps=1` so each iteration is straightforward for debugging.

In [ ]:
cfg = {
    "batch_size": 32,
    "seq_len": 12,
    "puzzle_emb_ndim": 0,
    "num_puzzle_identifiers": 1,
    "vocab_size": 16,
    "H_cycles": 1,
    "L_cycles": 1,
    "H_layers": 0,
    "L_layers": 1,
    "hidden_size": 64,
    "expansion": 2.0,
    "num_heads": 4,
    "pos_encodings": "rope",
    "halt_max_steps": 1,
    "halt_exploration_prob": 0.0,
    "forward_dtype": "float32",
    "mlp_t": False,
    "puzzle_emb_len": 0,
    "no_ACT_continue": True,
}

model = EdgeTRMAdapter(cfg, loss_type="stablemax_cross_entropy").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print("params:", sum(p.numel() for p in model.parameters()))

## 2) Synthetic task

Identity token reconstruction (`labels = inputs`).

If the training path is wired correctly, the loss should trend down quickly.

In [ ]:
def make_batch(batch_size=32, seq_len=12, vocab_size=16, device=device):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = x.clone()
    puzzle_ids = torch.zeros((batch_size,), dtype=torch.int32, device=device)
    return EdgeTRMBatch(inputs=x, labels=y, puzzle_identifiers=puzzle_ids)

In [ ]:
model.train()
losses = []
lm_losses = []
accs = []

# fixed batch so you can verify optimization wiring by overfitting
batch = make_batch(cfg["batch_size"], cfg["seq_len"], cfg["vocab_size"])
carry = model.initial_carry(batch)
# Move carry tensors to selected device
carry.inner_carry.z_H = carry.inner_carry.z_H.to(device)
carry.inner_carry.z_L = carry.inner_carry.z_L.to(device)
carry.steps = carry.steps.to(device)
carry.halted = carry.halted.to(device)
carry.current_data = {k: v.to(device) for k, v in carry.current_data.items()}

for step in range(300):
    carry, loss, metrics, _, done = model.train_step(batch=batch, carry=carry)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(float(loss.detach().cpu()))
    lm_losses.append(float((metrics["lm_loss"] / metrics["count"]).detach().cpu()))
    accs.append(float((metrics["accuracy"] / metrics["count"]).detach().cpu()))

    if step % 50 == 0:
        print(
            f"step={step:03d} total={losses[-1]:.4f} lm={lm_losses[-1]:.4f} tok_acc={accs[-1]:.4f}"
        )

print("start_lm_loss:", lm_losses[0], "end_lm_loss:", lm_losses[-1])
print("start_acc:", accs[0], "end_acc:", accs[-1])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(lm_losses)
ax[0].set_title("LM loss per token")
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].grid(alpha=0.2)

ax[1].plot(accs)
ax[1].set_title("Token accuracy")
ax[1].set_xlabel("step")
ax[1].set_ylabel("accuracy")
ax[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Interpreting these numbers vs the official paper

These logs are **not paper-level benchmark results**. They come from a tiny synthetic overfit sanity check (`labels = inputs`) used only to validate training wiring.

- `lm` and `tok_acc` are the useful sanity indicators here (they should improve).
- `total` includes ACT/Q halting terms from `ACTLossHead`, so it can fluctuate even when language-modeling behavior improves.
- The paper reports benchmark metrics on Sudoku/ARC setups with large training runs and specific dataset/eval protocols, not this toy identity task.

## 3) Next step for your real use-case

Replace `make_batch(...)` with your Sudoku/ARC batch function, while keeping:

- `EdgeTRMBatch(inputs, labels, puzzle_identifiers)`
- `carry = model.initial_carry(...)` once
- recurrent `carry, loss, metrics, ... = model.train_step(...)`

That keeps recursion + ACT loss behavior aligned with the official implementation.

## 4) Real Sudoku pipeline (wrapper-compatible)

Below are concrete, copy-ready steps adapted from `EdgeTRM.ipynb`, but wired to `EdgeTRMAdapter` / `EdgeTRMBatch`.

In [ ]:
import subprocess
from pathlib import Path
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# Build a small Sudoku dataset using the official TinyRecursiveModels builder
# (downloads from Hugging Face dataset used by the repo).
data_dir = Path('TinyRecursiveModels/data/sudoku-notebook')
if not data_dir.exists():
    cmd = [
        'python', 'dataset/build_sudoku_dataset.py',
        '--output-dir', 'data/sudoku-notebook',
        '--subsample-size', '2048',
        '--num-aug', '0',
    ]
    subprocess.run(cmd, cwd='TinyRecursiveModels', check=True)

print('dataset dir:', data_dir.resolve())

In [ ]:
# Load official-format arrays produced by the dataset builder
train_inputs = np.load('TinyRecursiveModels/data/sudoku-notebook/train/all__inputs.npy')
train_labels = np.load('TinyRecursiveModels/data/sudoku-notebook/train/all__labels.npy')
test_inputs  = np.load('TinyRecursiveModels/data/sudoku-notebook/test/all__inputs.npy')
test_labels  = np.load('TinyRecursiveModels/data/sudoku-notebook/test/all__labels.npy')

# Split train into train/val
n_train = int(0.85 * len(train_inputs))
X_train, Y_train = train_inputs[:n_train], train_labels[:n_train]
X_val, Y_val = train_inputs[n_train:], train_labels[n_train:]
X_test, Y_test = test_inputs, test_labels

def make_loader(X_np, Y_np, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X_np), torch.from_numpy(Y_np))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=True)

train_loader = make_loader(X_train, Y_train, batch_size=64, shuffle=True)
val_loader = make_loader(X_val, Y_val, batch_size=64, shuffle=False)
test_loader = make_loader(X_test, Y_test, batch_size=64, shuffle=False)

print('shapes:', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
def edge_batch_from_xy(x: torch.Tensor, y: torch.Tensor, device: torch.device):
    x = x.to(device=device, dtype=torch.int32)
    y = y.to(device=device, dtype=torch.int32)
    y = torch.where(y == 0, torch.full_like(y, -100), y)  # official ignore label
    puzzle_ids = torch.zeros((x.shape[0],), dtype=torch.int32, device=device)
    return EdgeTRMBatch(inputs=x, labels=y, puzzle_identifiers=puzzle_ids)


def make_sudoku_cfg(batch_size: int = 64):
    return {
        "batch_size": batch_size,
        "seq_len": 81,
        "puzzle_emb_ndim": 0,
        "num_puzzle_identifiers": 1,
        "vocab_size": 11,   # PAD + digits 0..9
        "H_cycles": 3,
        "L_cycles": 6,
        "H_layers": 0,
        "L_layers": 2,
        "hidden_size": 256,
        "expansion": 4.0,
        "num_heads": 8,
        "pos_encodings": "rope",
        "halt_max_steps": 8,
        "halt_exploration_prob": 0.1,
        "forward_dtype": "float32",
        "mlp_t": False,
        "puzzle_emb_len": 0,
        "no_ACT_continue": True,
    }

In [ ]:
sudoku_cfg = make_sudoku_cfg(batch_size=64)
sudoku_model = EdgeTRMAdapter(sudoku_cfg, loss_type="stablemax_cross_entropy").to(device)
sudoku_opt = torch.optim.AdamW(sudoku_model.parameters(), lr=1e-4, weight_decay=1.0, betas=(0.9, 0.95))

# Initialize carry once and keep recurrent updates across batches
init_x, init_y = next(iter(train_loader))
init_batch = edge_batch_from_xy(init_x, init_y, device)
carry = sudoku_model.initial_carry(init_batch)
carry.inner_carry.z_H = carry.inner_carry.z_H.to(device)
carry.inner_carry.z_L = carry.inner_carry.z_L.to(device)
carry.steps = carry.steps.to(device)
carry.halted = carry.halted.to(device)
carry.current_data = {k: v.to(device) for k, v in carry.current_data.items()}

In [ ]:
def run_epoch(model, loader, optimizer=None, carry=None):
    train_mode = optimizer is not None
    model.train(train_mode)

    lm_sum = acc_sum = count_sum = 0.0

    for x, y in loader:
        batch = edge_batch_from_xy(x, y, device)

        if carry is None:
            carry = model.initial_carry(batch)
            carry.inner_carry.z_H = carry.inner_carry.z_H.to(device)
            carry.inner_carry.z_L = carry.inner_carry.z_L.to(device)
            carry.steps = carry.steps.to(device)
            carry.halted = carry.halted.to(device)
            carry.current_data = {k: v.to(device) for k, v in carry.current_data.items()}

        with torch.set_grad_enabled(train_mode):
            carry, loss, metrics, _, _ = model.train_step(batch=batch, carry=carry)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        lm_sum += float(metrics["lm_loss"].detach().cpu())
        acc_sum += float(metrics["accuracy"].detach().cpu())
        count_sum += float(metrics["count"].detach().cpu())

    return {"lm": lm_sum / max(count_sum, 1.0), "tok_acc": acc_sum / max(count_sum, 1.0)}, carry

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    tr, carry = run_epoch(sudoku_model, train_loader, optimizer=sudoku_opt, carry=carry)
    va, carry = run_epoch(sudoku_model, val_loader, optimizer=None, carry=carry)
    print(f"epoch={epoch:02d} train_lm={tr['lm']:.4f} train_tok_acc={tr['tok_acc']:.4f} val_lm={va['lm']:.4f} val_tok_acc={va['tok_acc']:.4f}")

## 5) Official paper-style training command (from TinyRecursiveModels)

For exact paper-style protocol, run the repo pipeline directly (large run, not notebook quick-run):

In [ ]:
OFFICIAL_TRAIN_CMD = r'''
cd TinyRecursiveModels
python dataset/build_sudoku_dataset.py --output-dir data/sudoku-extreme-1k-aug-1000 --subsample-size 1000 --num-aug 1000
python pretrain.py \
  arch=trm \
  data_paths="[data/sudoku-extreme-1k-aug-1000]" \
  evaluators="[]" \
  epochs=50000 eval_interval=5000 \
  lr=1e-4 puzzle_emb_lr=1e-4 weight_decay=1.0 puzzle_emb_weight_decay=1.0 \
  arch.mlp_t=True arch.pos_encodings=none \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=6 \
  +run_name=trm_mlp_sudoku ema=True
'''
print(OFFICIAL_TRAIN_CMD)